In [ ]:
import pandas as pd

orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")


In [3]:
orders["flag_invalid_approval_carrier"] = (
    orders["order_approved_at"] > orders["order_delivered_carrier_date"]
)
print(orders["flag_invalid_approval_carrier"].sum())

# Found 1,359 orders where the recorded carrier handoff precedes approval.
# This may reflect the actual process or a delay in recording events; the cause is unconfirmed.
# Original timestamps were retained.



1359


In [4]:
orders["flag_invalid_carrier_delivery"] = (
    orders["order_delivered_carrier_date"]
    > orders["order_delivered_customer_date"]
)
print(orders["flag_invalid_carrier_delivery"].sum())

# theres 23 rows where the carrier delivery date is after the customer delivery date which is impossible

23


In [5]:
orders["flag_missing_delivery_date"] = (
    (orders["order_status"] == "delivered")
    & orders["order_delivered_customer_date"].isna()
)
print(orders["flag_missing_delivery_date"].sum())

# 8 rows have missing delivery dates despite having a status of delivered

8


In [6]:
orders["flag_status_delivery_mismatch"] = (
    (orders["order_status"] != "delivered")
    & orders["order_delivered_customer_date"].notna()
)
print(orders["flag_status_delivery_mismatch"].sum())

orders.loc[
    orders["flag_status_delivery_mismatch"],
    ["order_id", "order_status", "order_delivered_customer_date"]
]

### Order status and delivery date mismatch
# Found 6 orders marked `canceled` with a recorded customer delivery timestamp.
# The available fields do not establish whether the status or timestamp is incorrect.
# Retained the original values and flagged these rows using `flag_status_delivery_mismatch`.

6


,order_id,order_status,order_delivered_customer_date
2921,1950d777989f6a877539f53795b4c3c3,canceled,2018-03-21 22:03:51
8791,dabf2b0e35b423f94618bf965fcb7514,canceled,2016-10-16 14:36:59
58266,770d331c84e5b214bd9dc70a10b829d0,canceled,2016-10-14 15:07:11
59332,8beb59392e21af5eb9547ae1a9938d06,canceled,2016-10-19 18:47:43
92636,65d1e226dfaeb8cdc42f665422522d14,canceled,2016-11-08 10:58:34
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,canceled,2016-11-09 14:53:50


In [8]:
exclude_rows = (
    orders["flag_invalid_carrier_delivery"]
    | orders["flag_missing_delivery_date"]
    | orders["flag_status_delivery_mismatch"]
)

orders_clean = orders.loc[~exclude_rows].copy()

print("Rows before:", len(orders))
print("Rows removed:", exclude_rows.sum())
print("Rows after:", len(orders_clean))
orders_clean[[
    "flag_invalid_carrier_delivery",
    "flag_missing_delivery_date",
    "flag_status_delivery_mismatch",
]].sum()

Rows before: 99441
Rows removed: 37
Rows after: 99404


flag_invalid_carrier_delivery    0
flag_missing_delivery_date       0
flag_status_delivery_mismatch    0
dtype: int64

In [9]:
from pathlib import Path

processed_folder = Path("../data/processed")
processed_folder.mkdir(parents=True, exist_ok=True)

orders_clean.to_csv(processed_folder / "orders_clean.csv", index=False)